# Session 11: Regression, Ridge & Lasso

Practice companion for New. **Synthetic demonstration data**, not the lecturer's original dataset. Run the code cells top to bottom. The original slide snippets remain available separately in the website.

Requires numpy, pandas, matplotlib, seaborn, scikit-learn, and plotly. Install missing packages with `%pip install numpy pandas matplotlib seaborn scikit-learn plotly`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
X, y = make_regression(n_samples=240, n_features=5, noise=25, random_state=42)
X = pd.DataFrame(X, columns=['size', 'marketing', 'footfall', 'staff', 'competitors'])
X.iloc[::17, 0] = np.nan
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
def pipeline(model):
    return Pipeline([('impute', SimpleImputer(strategy='median')),
                     ('scale', StandardScaler()), ('model', model)])
# Search the WHOLE pipeline so preprocessing is learned separately within each fold.
search = GridSearchCV(pipeline(LinearRegression()), [
    {'model': [LinearRegression()]},
    {'model': [Ridge()], 'model__alpha': [0.1, 1, 10, 100]},
    {'model': [Lasso(max_iter=10000)], 'model__alpha': [0.01, 0.1, 1, 10]},
], cv=5, scoring='neg_root_mean_squared_error')
search.fit(X_train, y_train)
for params, score in zip(search.cv_results_['params'], search.cv_results_['mean_test_score']):
    print(params, 'CV RMSE:', round(-score, 3))
print('Chosen on training CV:', search.best_params_)
preds = search.predict(X_test)
print('Final held-out RMSE:', np.sqrt(mean_squared_error(y_test, preds)))
print('Final held-out R2:', r2_score(y_test, preds))


## Your turn
Compare OLS and Ridge using the same folds and error metric. Report the measured difference; improvement is not guaranteed.

## Watch out
A coefficient is not automatically a causal effect, and Lasso's selected features are not guaranteed to be the true drivers. Ridge does not guarantee every coefficient is nonzero.